# GSB 5544 — PA 3.2: Distances Between Observations

---
title: "PA 3.2: Distances Between Observations"
format:
  html:
    embed-resources: true
---

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

The Ames data set (2,930 home sales in Ames, Iowa; tab-separated) is at the URL below. House 0 is the first row. The variables we need for part 1: `Gr Liv Area` (above-ground living area, sq ft), `Bedroom AbvGr`, `Full Bath`, `Half Bath`, and `SalePrice`; part 2 adds `House Style`.

In [2]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")

df_ames.shape

(2930, 82)

In [3]:
df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [4]:
df_ames.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

In [5]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 *df_ames["Half Bath"]

house0 = df_ames.loc[0]
show_vars = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "Year Built", "SalePrice"]
house0[show_vars]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           1.0
House Style      1Story
Neighborhood      NAmes
Year Built         1960
SalePrice        215000
Name: 0, dtype: object

In [6]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X = df_ames[features].astype(float)

In [7]:
X_z = (X - X.mean()) / X.std()    

# taking each z score and subtracting it by the first house
diff = X_z - X_z.loc[0]

In [8]:
df_ames["dist_euclid"] = np.sqrt((diff **2).sum(axis=1))
df_ames["dist_manhattan"] = diff.abs().sum(axis=1)

In [9]:
# find the price of the houses that's cheaper than the first house
df_cheaper_sales_price = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]

In [ ]:
# euclidean distance

In [10]:
df_cheaper_sales_price.sort_values("dist_euclid")[show_vars + ["dist_euclid"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_euclid
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.017804
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.019782
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.019782
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.019782


In [ ]:
# manhattan distance

In [11]:
df_cheaper_sales_price.sort_values("dist_manhattan")[show_vars + ["dist_manhattan"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_manhattan
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.017804
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.019782
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.019782
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.019782


**YOUR RESPONSE HERE** We added sale price after finding the home with the closest distance (euclidean and manhattan). The results we found were similar considering the three variables we selected

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [12]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * df_ames["Half Bath"]

house0 = df_ames.loc[0]

show_vars = [
    "Gr Liv Area",
    "Bedroom AbvGr",
    "Bathrooms",
    "House Style",
    "Neighborhood",
    "Year Built",
    "SalePrice"]

house0[show_vars]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           1.0
House Style      1Story
Neighborhood      NAmes
Year Built         1960
SalePrice        215000
Name: 0, dtype: object

For part 2, I used living area, bedroom, bathrooms, and house style. The homes with the smallest distances were typically similar to house 0 in all categories, so the recommendations work.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [13]:
quant_features = [
    "Gr Liv Area",
    "Bedroom AbvGr",
    "Bathrooms",
    "Year Built",
    "Garage Cars"]

In [14]:
cat_features = [
    "House Style",
    "Neighborhood"]

In [15]:
X = df_ames[quant_features].astype(float)

X_z = (X - X.mean()) / X.std()

In [16]:
X_cat = pd.get_dummies(
    df_ames[cat_features],
    dtype=float)

In [17]:
X_all = pd.concat([X_z, X_cat], axis=1)

In [18]:
diff_all = X_all - X_all.loc[0]

df_ames["dist_euclid_3"] = np.sqrt(
    (diff_all ** 2).sum(axis=1))

df_ames["dist_manhattan_3"] = (
    diff_all.abs().sum(axis=1))

In [19]:
df_cheaper_3 = df_ames[
    df_ames["SalePrice"] < house0["SalePrice"]]

In [20]:
show_vars_3 = [
    "Gr Liv Area",
    "Bedroom AbvGr",
    "Bathrooms",
    "House Style",
    "Neighborhood",
    "Year Built",
    "Garage Cars",
    "SalePrice"]

In [21]:
df_cheaper_3.sort_values("dist_euclid_3")[
    show_vars_3 + ["dist_euclid_3"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,Garage Cars,SalePrice,dist_euclid_3
1240,1570,3,1.0,1Story,NAmes,1958,2.0,166800,0.182525
618,1644,3,1.0,1Story,NAmes,1953,2.0,167000,0.232655
2558,1433,3,1.0,1Story,NAmes,1961,2.0,161000,0.442377
1896,1429,3,1.0,1Story,NAmes,1960,2.0,181900,0.449052
989,1414,3,1.0,1Story,NAmes,1958,2.0,176500,0.483271


In [22]:
df_cheaper_3.sort_values("dist_manhattan_3")[
    show_vars_3 + ["dist_manhattan_3"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,Garage Cars,SalePrice,dist_manhattan_3
1240,1570,3,1.0,1Story,NAmes,1958,2.0,166800,0.236251
618,1644,3,1.0,1Story,NAmes,1953,2.0,167000,0.255179
1896,1429,3,1.0,1Story,NAmes,1960,2.0,181900,0.449052
2558,1433,3,1.0,1Story,NAmes,1961,2.0,161000,0.474203
989,1414,3,1.0,1Story,NAmes,1958,2.0,176500,0.544851


The houses I found using manhattan distance seem to make sense as an alternative to house 0. Each of the five closest houses are 1 story, in the right neighborhood, 3 bedrooms, 1 bathroom, and 2 garage spaces.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [23]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [24]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [26]:
features = ["AdmissionRate", "Undergraduates"]

X = df_college[features].astype(float)

In [27]:
X_z = (X - X.mean()) / X.std()

In [28]:
diff = X_z - X_z.loc[school_name]

In [29]:
df_college["dist_euclid"] = np.sqrt(
    (diff ** 2).sum(axis=1))

In [30]:
df_college.sort_values("dist_euclid")[
    ["AdmissionRate", "Undergraduates", "dist_euclid"]].head(6)

,AdmissionRate,Undergraduates,dist_euclid
Institution,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,0.000000
University of California-Santa Barbara,0.2918,23081.0,0.309162
DeVry University-Illinois,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846
Clemson University,0.4922,21577.0,0.736788
University of Virginia-Main Campus,0.2074,17041.0,0.761296


The schools with the smallest distances are the most similar. In this case, Univ. of California, Devry, and UNC are the most similar to Cal Poly in regards to undergrads and each college's acceptance rate

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [31]:
features = ["AdmissionRate", "Undergraduates"]

X = df_college[features].astype(float)

X_z = (X - X.mean()) / X.std()

In [33]:
X_cat = pd.get_dummies(
    df_college[["CarnegieClassification", "Ownership"]],
    dtype=float)

In [34]:
X_all = pd.concat([X_z, X_cat], axis=1)

In [35]:
diff = X_all - X_all.loc[school_name]

df_college["dist_euclid_2"] = np.sqrt(
    (diff ** 2).sum(axis=1))

In [36]:
df_college.sort_values("dist_euclid_2")[
    ["AdmissionRate", "Undergraduates", "CarnegieClassification", "Ownership", "dist_euclid_2"]].head(6)

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist_euclid_2
Institution,,,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000000
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447612


After standardizing and calculating the distance from Cal Poly, I found the most similar schools to be CUNY Hunter, CUNY Bernard, CUNY John Jay.... followed by two other universities. These results make sense b/c the closest schools are public and have a similar Carnegie classification

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [37]:
pcip_cols = [col for col in df_college.columns if col.startswith("PCIP")]

In [38]:
X = df_college[pcip_cols].astype(float)

In [39]:
target = X.loc[school_name]

In [40]:
norms = np.sqrt((X ** 2).sum(axis=1))

In [41]:
df_college["cosine"] = (
    (X @ target) / (norms * norms[school_name]))

In [42]:
df_college["cosine"] = (
    (X @ target) / (norms * norms[school_name]))

In [43]:
df_college.sort_values("cosine", ascending=False)[
    ["cosine"]].head(6)

,cosine
Institution,
California Polytechnic State University-San Luis Obispo,1.000000
North Carolina State University at Raleigh,0.968572
Iowa State University,0.967254
University of Illinois Urbana-Champaign,0.941258
Mississippi State University,0.932753
Texas A & M University-College Station,0.926809


Cal Poly's closest cosine similarities ranged from 0.927 to 0.969. These values tell us that most schools have a similar distribution across various academic fields. Cosine similarity makes sense here b/c the goal was to compare the mix of majors rather than differences in size